# Compare Case Studies

Compare training curves and evaluation metrics across multiple case studies
(e.g., SF Bay vs Puget Sound).

**Usage:** Edit the `CASE_STUDIES` list below.

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

# --- Configuration ---
CASE_STUDIES = [
    '../case_studies/sf_bay',
    # '../case_studies/puget_sound',  # uncomment when available
]

case_dirs = {Path(cs).name: Path(cs) for cs in CASE_STUDIES}
print('Case studies:', list(case_dirs.keys()))

## 1. Load evaluation results

In [ ]:
results = {}

for name, cdir in case_dirs.items():
    results_path = cdir / 'outputs' / 'evaluation' / 'test_results.json'
    if results_path.exists():
        with open(results_path) as f:
            results[name] = json.load(f)
        print(f'{name}: loaded (epoch {results[name].get("checkpoint_epoch", "?")})')
    else:
        print(f'{name}: no evaluation results found at {results_path}')

if not results:
    print('\nNo results to compare. Run evaluate.py for at least one case study.')

## 2. Metrics comparison table

In [ ]:
if results:
    # Collect all metric keys
    all_metrics = set()
    for r in results.values():
        all_metrics.update(r.get('metrics', {}).keys())
    all_metrics = sorted(all_metrics)

    # Header
    header = f'{"Metric":<20}'
    for name in results:
        header += f'{name:>15}'
    print(header)
    print('-' * len(header))

    # Test loss
    row = f'{"test_loss":<20}'
    for name in results:
        val = results[name].get('test_loss', float('nan'))
        row += f'{val:>15.4f}'
    print(row)

    # Per-metric rows
    for metric in all_metrics:
        row = f'{metric:<20}'
        for name in results:
            val = results[name].get('metrics', {}).get(metric, float('nan'))
            row += f'{val:>15.4f}'
        print(row)

## 3. Metrics bar chart

In [ ]:
if results and all_metrics:
    names = list(results.keys())
    x = np.arange(len(all_metrics))
    width = 0.8 / max(len(names), 1)

    fig, ax = plt.subplots(figsize=(12, 5))

    for i, name in enumerate(names):
        vals = [results[name].get('metrics', {}).get(m, 0) for m in all_metrics]
        ax.bar(x + i * width, vals, width, label=name)

    ax.set_xticks(x + width * (len(names) - 1) / 2)
    ax.set_xticklabels(all_metrics, rotation=30, ha='right')
    ax.set_ylabel('Value')
    ax.set_title('Metric Comparison Across Case Studies')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

## 4. Training loss curves

Compare training and validation loss curves from TensorBoard event files.

In [ ]:
# Check for saved training_loss.png from training script
fig, axes = plt.subplots(1, len(case_dirs), figsize=(7 * len(case_dirs), 5))
if len(case_dirs) == 1:
    axes = [axes]

for ax, (name, cdir) in zip(axes, case_dirs.items()):
    loss_img = cdir / 'checkpoints' / 'training_loss.png'
    if loss_img.exists():
        img = plt.imread(str(loss_img))
        ax.imshow(img)
        ax.set_title(name)
        ax.axis('off')
    else:
        ax.text(0.5, 0.5, f'No training_loss.png\nfor {name}',
                ha='center', va='center', transform=ax.transAxes)
        ax.set_title(name)

plt.suptitle('Training Loss Curves', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Configuration comparison

Side-by-side view of key hyperparameters.

In [ ]:
import yaml

configs = {}
for name, cdir in case_dirs.items():
    cfg_path = cdir / 'configs' / 'training.yaml'
    if cfg_path.exists():
        with open(cfg_path) as f:
            configs[name] = yaml.safe_load(f)

# Key hyperparameters to compare
keys = ['base_channels', 'sequence_length', 'batch_size', 'num_epochs',
        'learning_rate', 'weight_decay', 'loss_alpha', 'loss_beta', 'loss_gamma']

if configs:
    header = f'{"Parameter":<25}'
    for name in configs:
        header += f'{name:>15}'
    print(header)
    print('-' * len(header))

    for key in keys:
        row = f'{key:<25}'
        for name in configs:
            val = configs[name].get(key, 'N/A')
            row += f'{str(val):>15}'
        print(row)